In [1]:
!pip install -q bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 49.5 MB/s eta 0:00:00


In [2]:
import torch
import random
import os
import json
import glob
from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoProcessor, 
    Qwen3VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig
)
import re
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Dict, List

# --- 1. CẤU HÌNH CƠ BẢN ---
model_id = "Qwen/Qwen3-VL-2B-Instruct" 
output_dir = "/kaggle/working/qwen-vl-lora-vizwiz"
OUTPUT_DIR = "/kaggle/working" # Thêm biến này cho phần lưu JSON

SYSTEM_PROMPT = """Bạn là một trợ lý AI xuất sắc trong việc phân tích hình ảnh (VQA).
Nhiệm vụ của bạn là trả lời câu hỏi của người dùng dựa trên hình ảnh được cung cấp một cách ngắn gọn, chính xác bằng Tiếng Việt một cách tự nhiên.

QUY TẮC QUAN TRỌNG:
1. Bạn phải CỐ GẮNG HẾT SỨC để trả lời dựa trên bất kỳ chi tiết nào bạn có thể nhận diện được trong ảnh,một cách ngắn gọn , xúc tích.
2. CHỈ KHI hình ảnh THỰC SỰ QUÁ MỜ, nhòe, nhiễu nặng hoặc tối đen đến mức MỘT CON NGƯỜI cũng không thể đoán được bất cứ thông tin gì liên quan đến câu hỏi,thì bạn phải trả lời đúng một câu: "Ảnh mờ, bạn vui lòng chụp lại ảnh rõ hơn."
3. Nếu ảnh chỉ hơi mờ hoặc thiếu sáng nhưng vẫn có thể suy luận được, tuyệt đối không phàn nàn, hãy đưa ra câu trả lời tốt nhất của bạn.
4. Trả lời ngắn gọn, súc tích trong một câu, không giải thích dài dòng, không thêm thông tin không cần thiết, chỉ trả lời đúng trọng tâm câu hỏi.
5. Luôn trả lời bằng tiếng Việt, không được trả lời bằng ngôn ngữ khác, không được trả lời bằng tiếng Anh, không được trả lời bằng tiếng Trung, chỉ được trả lời bằng tiếng Việt.
"""

In [3]:
# --- 2. XỬ LÝ DỮ LIỆU ---
random.seed(168)
IMAGE_DIR_0 = "/kaggle/input/datasets/thanhngphu/vizwiz-translated-vn/vizwiz_dataset_1000/answerable_0"
IMAGE_DIR_1 = "/kaggle/input/datasets/thanhngphu/vizwiz-translated-vn/vizwiz_dataset_1000/answerable_1"
JSON_0 = "/kaggle/input/datasets/thanhngphu/vizwiz-translated-vn/vizwiz_dataset_1000/pixtral_ans_0_translated.json"
JSON_1 = "/kaggle/input/datasets/thanhngphu/vizwiz-translated-vn/vizwiz_dataset_1000/pixtral_ans_1_translated.json"

all_data = []

def process_file(json_file, img_dir):
    valid_images = set(os.listdir(img_dir))
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for item in data:
        image_name = item.get("image", "").strip()
        question_vi = item.get("question", "").strip()
        answer_vi = item.get("AI_answer", "").strip()
        
        if not image_name.lower().endswith(".jpg"): continue
        if image_name not in valid_images or not question_vi or not answer_vi: continue
            
        all_data.append({
            "images": [os.path.join(img_dir, image_name)],
            "messages": [
                {"role": "user", "content": f"<image>\n{question_vi}"},
                {"role": "assistant", "content": answer_vi}
            ]
        })

process_file(JSON_0, IMAGE_DIR_0)
process_file(JSON_1, IMAGE_DIR_1)

random.shuffle(all_data)
n = len(all_data)
train_end = int(0.7 * n)
val_end = int(0.8 * n)

def save_json(data, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

save_json(all_data[:train_end], "vizwiz_train.json")
save_json(all_data[train_end:val_end], "vizwiz_val.json")
save_json(all_data[val_end:], "vizwiz_test.json")
print(f"Hoàn tất chia tập dữ liệu! (Train: {train_end}, Val: {val_end - train_end}, Test: {n - val_end})")

# --- Load lại thành format Dataset của HuggingFace ---
hf_train_dataset = load_dataset("json", data_files=os.path.join(OUTPUT_DIR, "vizwiz_train.json"), split="train")
hf_val_dataset = load_dataset("json", data_files=os.path.join(OUTPUT_DIR, "vizwiz_val.json"), split="train")


Hoàn tất chia tập dữ liệu! (Train: 700, Val: 100, Test: 200)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
# --- 3. LOAD PROCESSOR VÀ MODEL ---
print("Đang load processor và model ở dạng bfloat16...")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation='sdpa',
)

model.enable_input_require_grads()

Đang load processor và model ở dạng bfloat16...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

In [5]:
print("=== 1. CÁC MODULE CHÍNH (TOP-LEVEL) ===")
for name, module in model.named_children():
    print(f" - {name}")

print("\n=== 2. CẤU TRÚC TIỀN TỐ (PREFIX) CỦA CÁC THAM SỐ ===")
# Dùng Set để gom nhóm các tên bị trùng lặp
prefixes = set()
for name, param in model.named_parameters():
    # Lấy 2 cấp đầu tiên của tên tham số (VD: 'visual.blocks', 'model.layers')
    parts = name.split('.')
    if len(parts) >= 2:
        prefix = f"{parts[0]}.{parts[1]}"
    else:
        prefix = parts[0]
    prefixes.add(prefix)

# In ra theo thứ tự abc
for prefix in sorted(prefixes):
    print(f" + {prefix}.*")

=== 1. CÁC MODULE CHÍNH (TOP-LEVEL) ===
 - model
 - lm_head

=== 2. CẤU TRÚC TIỀN TỐ (PREFIX) CỦA CÁC THAM SỐ ===
 + model.language_model.*
 + model.visual.*


In [6]:
# --- 4. CẤU HÌNH LORA (CHỈ FINETUNE LLM, FREEZE VISION ENCODER) ---

# 1. Cấu hình LoRA: Dùng Regex để ép LoRA chỉ gắn vào các lớp của Language Model
lora_config = LoraConfig(
    r=64, 
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Gắn LoRA adapter vào model
model = get_peft_model(model, lora_config)

# 3. Đóng băng (Freeze) toàn bộ Vision Encoder một cách tuyệt đối
print("Đang rà soát và đóng băng Vision Encoder...")
for name, param in model.named_parameters():
    # Trong cấu trúc Qwen-VL, phần xử lý ảnh thường nằm trong module 'visual'
    if "visual" in name:
        param.requires_grad = False

# 4. In ra thông số để kiểm tra
model.print_trainable_parameters()

Đang rà soát và đóng băng Vision Encoder...
trainable params: 69,730,304 || all params: 2,197,262,336 || trainable%: 3.1735


In [7]:
# --- 5. TẠO CUSTOM DATASET CHO QWEN-VL ---
class QwenVLDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        img_path = item['images'][0]
        question = item['messages'][0]['content'].replace("<image>", "").strip()
        answer = item['messages'][1]['content'].strip()

        # 1. Tạo cấu trúc messages chuẩn của Qwen CÓ GIỚI HẠN PIXEL
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image", 
                        "image": img_path,
                        "max_pixels": 409600 
                    }, 
                    {"type": "text", "text": question}
                ]
            }
        ]
        
        # 2. Xử lý prompt text và trích xuất ảnh
        prompt_text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        # 3. Dùng Processor xử lý thành tensor
        prompt_inputs = self.processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            return_tensors="pt",
            padding=False
        )

        # 4. Tokenize câu trả lời (Ground Truth)
        answer_ids = self.processor.tokenizer(
            answer + self.processor.tokenizer.eos_token, 
            return_tensors="pt", 
            add_special_tokens=False
        ).input_ids

        # 5. Ghép Prompt và Answer lại thành chuỗi input hoàn chỉnh
        input_ids = torch.cat([prompt_inputs.input_ids[0], answer_ids[0]])
        
        # 6. Tạo Label: Ẩn (-100) phần prompt, chỉ tính loss trên phần Answer
        labels = torch.cat([
            torch.full_like(prompt_inputs.input_ids[0], -100),
            answer_ids[0]
        ])

        return {
            "input_ids": input_ids,
            "labels": labels,
            "pixel_values": prompt_inputs.pixel_values,
            "image_grid_thw": prompt_inputs.image_grid_thw
        }

train_dataset = QwenVLDataset(hf_train_dataset, processor)
val_dataset = QwenVLDataset(hf_val_dataset, processor)

In [8]:
# --- 6. DATA COLLATOR RIÊNG CHO QWEN-VL ---
@dataclass
class QwenVLDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]
        
        pixel_values = torch.cat([f["pixel_values"] for f in features], dim=0)
        image_grid_thw = torch.cat([f["image_grid_thw"] for f in features], dim=0)

        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.processor.tokenizer.pad_token_id
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )
        attention_mask = input_ids.ne(self.processor.tokenizer.pad_token_id)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "pixel_values": pixel_values,
            "image_grid_thw": image_grid_thw
        }

In [9]:
import os
num_workers = os.cpu_count()
print(num_workers)

4


In [10]:
# --- 7. CẤU HÌNH TRAINING VÀ CHẠY ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,      
    per_device_eval_batch_size=2,       
    gradient_accumulation_steps=4,      
    learning_rate=2e-4,
    save_strategy="epoch",
    eval_strategy="epoch",              
    logging_strategy="epoch",
    dataloader_num_workers=num_workers,
    dataloader_pin_memory=True,
    num_train_epochs=3,                
    # bf16=True,                          
    report_to="none",
    load_best_model_at_end=True,
    save_total_limit=2,
    remove_unused_columns=False 
)

import gc
torch.cuda.empty_cache()
gc.collect()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=QwenVLDataCollator(processor=processor)
)

In [11]:
print("Bắt đầu training và đánh giá...")
trainer.train()

Bắt đầu training và đánh giá...


Epoch,Training Loss,Validation Loss
1,1.317243,1.157501
2,0.775180,1.206198
3,0.431650,1.278716


TrainOutput(global_step=264, training_loss=0.8413575345819647, metrics={'train_runtime': 36727.1649, 'train_samples_per_second': 0.057, 'train_steps_per_second': 0.007, 'total_flos': 1.805341452750029e+16, 'train_loss': 0.8413575345819647, 'epoch': 3.0})

In [12]:
# --- 8. LƯU MODEL ---
trainer.model.save_pretrained(f"{output_dir}/final_adapter")
processor.save_pretrained(f"{output_dir}/final_adapter")
print(f"Đã lưu LoRA adapter tại: {output_dir}/final_adapter")

Đã lưu LoRA adapter tại: /kaggle/working/qwen-vl-lora-vizwiz/final_adapter
